In [ ]:
import glob
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np
import re


# Ruta a tu carpeta con los datos
ruta = "/home/luciam/Documents/starry/Orion_yceb/dat_orion_sergio/sample_3/dat"  # cámbiala por la ruta real

# Lista de archivos ASCII
archivos = glob.glob(os.path.join(ruta, "*.dat"))  # o "*.dat" según corresponda
print(f"Se encontraron {len(archivos)} archivos")#aqui podemos corroborar que si es la carpeta/archivo que queremos leer


In [ ]:
dataframes = []

for archivo in archivos:
    try:
        # Leer archivo ASCII, ignorando comentarios
        datos = np.loadtxt(archivo, comments="#", skiprows=0)  # ajusta skiprows si hay cabecera
        if datos.shape[1] < 2:
            continue  # Necesitamos al menos 2 columnas
        
        df = pd.DataFrame({
            "time": datos[:,0],
            "magnitude": datos[:,1],
            "Archivo": os.path.basename(archivo)
        })
        dataframes.append(df)#guardamos en el diccionario dataframes las variables tiempo y magnitud
    except Exception as e:
        print(f"No se pudo leer {archivo}: {e}")

# Recorremos la lista de DataFrames directamente
for i, df in enumerate(dataframes):
    plt.figure(figsize=(10,5))
    plt.plot(df["time"], df["magnitude"], '.', markersize=3)
    plt.gca().invert_yaxis()  # Magnitudes más brillantes arriba, flip axis 
    plt.xlabel("Tiempo [d]")
    plt.ylabel("Magnitud")
    plt.title(f"Curve {i}: {df['Archivo'].iloc[0]}")
    plt.show()

In [ ]:
def extraer_objeto_sector(nombre_archivo):
    # Ejemplo nombre: TIC_252854439_S_86_LC.dat
    # Queremos: objeto = TIC_252854439, sector = 86
    match = re.search(r"(TIC_\d+)_S_(\d+)_LC", nombre_archivo)#Aqui separamos el nombre del objeto y del sector
    if match:
        objeto = match.group(1)#guardar el nombre
        sector = int(match.group(2))#guardar el sector
        return objeto, sector
    else:
        return None, None


In [ ]:
dataframes = []
# time magnitude magnitude_err flux_ap_no_corr flux_annulus
for archivo in archivos:
    try:
        datos = np.loadtxt(archivo, comments="#", skiprows=0)
        if datos.shape[1] < 2:
            continue

        objeto, sector = extraer_objeto_sector(os.path.basename(archivo))# tomamos como referencia el nombre base del archivo
        if objeto is None:
            continue

        df = pd.DataFrame({ #Guardamos cada uno de los componentes del diccionario
            "time": datos[:,0],
            "magnitude": datos[:,1],
            "Archivo": os.path.basename(archivo), #adicionamos
            "Objeto": objeto,
            "Sector": sector
        })
        dataframes.append(df)
    except Exception as e:
        print(f"No se pudo leer {archivo}: {e}")

In [ ]:
def extraer_objeto_sector(nombre_archivo):
    # Ejemplo nombre: TIC_252854439_S_86_LC.dat
    # Queremos: objeto = TIC_252854439, sector = 86
    match = re.search(r"(TIC_\d+)_S_(\d+)_LC", nombre_archivo)
    if match:
        objeto = match.group(1)
        sector = int(match.group(2))
        return objeto, sector
    else:
        return None, None 

dataframes = []

for archivo in archivos:
    try:
        datos = np.loadtxt(archivo, comments="#", skiprows=0)
        if datos.shape[1] < 2:
            continue

        objeto, sector = extraer_objeto_sector(os.path.basename(archivo))
        if objeto is None:
            continue

        # Magnitud → Flujo
        magnitudes = datos[:,1]
        flux = 10 ** (-magnitudes/2.5)

        df = pd.DataFrame({
            "time": datos[:,0],
            "magnitude": magnitudes,
            "flux": flux,  # nueva columna con flujo
            "Archivo": os.path.basename(archivo),
            "Objeto": objeto,
            "Sector": sector
        })
        dataframes.append(df)
    except Exception as e:
        print(f"No se pudo leer {archivo}: {e}")

# Agrupar por objeto
objetos = set(df["Objeto"].iloc[0] for df in dataframes)

colores = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

for objeto in objetos:
    dfs_objeto = [df for df in dataframes if df["Objeto"].iloc[0] == objeto]
    dfs_objeto = sorted(dfs_objeto, key=lambda x: x["Sector"].iloc[0])

    n_sectores = len(dfs_objeto)
    fig, axs = plt.subplots(1, n_sectores, figsize=(5*n_sectores,4), sharey=True)
    
    if n_sectores == 1:
        axs = [axs]

    for j, (ax, df) in enumerate(zip(axs, dfs_objeto)):
        ax.plot(df["time"], df["flux"], '.', markersize=2,
                color=colores[j % len(colores)], alpha=0.7)
        sector = df["Sector"].iloc[0]
        ax.set_title(f"Sector {sector}")
        ax.set_xlabel("Tiempo [d]")
        ax.set_ylabel("Flujo relativo (10^(-m/2.5))")

    plt.suptitle(f"Objeto {objeto} - todos los sectores (Flujo)")
    plt.tight_layout()
    plt.show()

In [ ]:
# Agrupar por objeto
objetos = set(df["Objeto"].iloc[0] for df in dataframes)  # todos los objetos

colores = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

for objeto in objetos:
    # Filtrar todos los DataFrames de ese objeto
    dfs_objeto = [df for df in dataframes if df["Objeto"].iloc[0] == objeto]
    
    # Ordenar por sector
    dfs_objeto = sorted(dfs_objeto, key=lambda x: x["Sector"].iloc[0])
    
    n_sectores = len(dfs_objeto)
    fig, axs = plt.subplots(1, n_sectores, figsize=(5*n_sectores,4), sharey=True)
    
    if n_sectores == 1:
        axs = [axs]  # para iterar aunque haya un solo subplot

    # Ploteamos cada sector en su subplot
    for j, (ax, df) in enumerate(zip(axs, dfs_objeto)):
        ax.plot(df["time"], df["magnitude"], '.', markersize=2,
                color=colores[j % len(colores)], alpha=0.7)
        ax.invert_yaxis()
        sector = df["Sector"].iloc[0]
        ax.set_title(f"Sector {sector}")
        ax.set_xlabel("Tiempo [d]")
        ax.set_ylabel("Magnitud")

    plt.suptitle(f"Objeto {objeto} - todos los sectores")
    plt.tight_layout()
    plt.show()

In [ ]:
# Agrupar por objeto
objetos = set(df["Objeto"].iloc[0] for df in dataframes)

colores = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

for objeto in objetos:
    dfs_objeto = [df for df in dataframes if df["Objeto"].iloc[0] == objeto]
    dfs_objeto = sorted(dfs_objeto, key=lambda x: x["Sector"].iloc[0])
    
    n_sectores = len(dfs_objeto)
    fig, axs = plt.subplots(1, n_sectores, figsize=(5*n_sectores,4), sharey=True)
    
    if n_sectores == 1:
        axs = [axs]

    for j, (ax, df) in enumerate(zip(axs, dfs_objeto)):
        # Flujo normalizado
        flux_norm = df["flux"] / np.nanmedian(df["flux"])

        ax.plot(df["time"], flux_norm, '.', markersize=2,
                color=colores[j % len(colores)], alpha=0.7)
        sector = df["Sector"].iloc[0]
        ax.set_title(f"Sector {sector}")
        ax.set_xlabel("Tiempo [d]")
        ax.set_ylabel("Flujo normalizado")

    plt.suptitle(f"Objeto {objeto} - todos los sectores (Flujo normalizado)")
    plt.tight_layout()
    plt.show()